# Stage 3: Real-Time Streaming Layer, AgriFlow Farming

## Kafka Topic Design

AgriFlow operates thousands of IoT sensors across farm fields, generating
continuous readings for soil moisture, temperature, and humidity. We designed
two Kafka topics to handle this real-time data:

| Topic | Event Type | Partitions | Rationale |
|---|---|---|---|
| `sensor-readings` | Soil/temp/humidity IoT readings per field | 3 | Partitioned by farm_id to parallelize processing across farm zones |
| `irrigation-alerts` | High water usage or drought-risk triggers | 1 | Low volume, single alert consumer is sufficient |

## Architecture Decision: Lambda vs Kappa

We chose **Lambda Architecture**:
- **Batch layer (Stage 2):** Spark processes historical sensor and crop data
  to answer long-term yield and irrigation questions with full accuracy
- **Streaming layer (Stage 3):** Kafka + Spark Structured Streaming handles
  real-time sensor alerts for immediate irrigation decisions

Lambda is the right fit for AgriFlow because farm analysts need both:
historical yield reports (batch) AND real-time drought alerts (streaming).

In [1]:
import subprocess
subprocess.run(["pip", "install", "kafka-python"], check=True)

CompletedProcess(args=['pip', 'install', 'kafka-python'], returncode=0)

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("agriflow-stage3-streaming") \
    .master("local[*]") \
    .config("spark.hadoop.fs.defaultFS", "hdfs://namenode:9000") \
    .config("spark.pyspark.python", "/opt/conda/bin/python3") \
    .config("spark.pyspark.driver.python", "/opt/conda/bin/python3") \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0") \
    .getOrCreate()

print("Spark connected:", spark.version)

Spark connected: 3.5.0


In [3]:
from kafka.admin import KafkaAdminClient, NewTopic
from kafka.errors import TopicAlreadyExistsError

admin = KafkaAdminClient(bootstrap_servers="kafka:9092")

topics = [
    NewTopic(name="sensor-readings",    num_partitions=3, replication_factor=1),
    NewTopic(name="irrigation-alerts",  num_partitions=1, replication_factor=1),
]

try:
    admin.create_topics(topics)
    print("Topics created: sensor-readings, irrigation-alerts")
except TopicAlreadyExistsError:
    print("Topics already exist, that's fine")

admin.close()

Topics already exist, that's fine


In [4]:
import json, time
from kafka import KafkaProducer
from datetime import datetime

producer = KafkaProducer(
    bootstrap_servers="kafka:9092",
    value_serializer=lambda v: json.dumps(v, default=str).encode("utf-8")
)

df_raw = spark.read \
    .option("multiLine", True) \
    .json("file:///home/jovyan/data/agriflow-raw/soil-sensors.json.gz") \
    .select("farm_id", "field_id", "sensor_id",
            "conditions.soil_moisture_pct",
            "conditions.soil_temp_f",
            "nutrients.ph_level") \
    .limit(100) \
    .toPandas()

print(f"Sending {len(df_raw)} events...")
print(f"Columns: {list(df_raw.columns)}")

for _, row in df_raw.iterrows():
    event = {
        "farm_id": str(row["farm_id"]),
        "field_id": str(row["field_id"]),
        "sensor_id": str(row["sensor_id"]),
        "soil_moisture": float(row["soil_moisture_pct"]) if row["soil_moisture_pct"] else 0.0,
        "soil_temp": float(row["soil_temp_f"]) if row["soil_temp_f"] else 0.0,
        "soil_ph": float(row["ph_level"]) if row["ph_level"] else 0.0,
        "event_time": datetime.now().isoformat()
    }
    producer.send("sensor-readings", event)
    producer.flush()
    time.sleep(0.2)

print("Done sending.")

Sending 100 events...
Columns: ['farm_id', 'field_id', 'sensor_id', 'soil_moisture_pct', 'soil_temp_f', 'ph_level']
Done sending.


In [5]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

schema = StructType([
    StructField("farm_id", StringType()),
    StructField("field_id", StringType()),
    StructField("sensor_id", StringType()),
    StructField("soil_moisture", DoubleType()),
    StructField("soil_temp", DoubleType()),
    StructField("soil_ph", DoubleType()),
    StructField("event_time", StringType()),
])

raw_stream = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:9092") \
    .option("subscribe", "sensor-readings") \
    .option("startingOffsets", "latest") \
    .load()

parsed = raw_stream \
    .select(F.from_json(F.col("value").cast("string"), schema).alias("data")) \
    .select("data.*") \
    .withColumn("event_time", F.to_timestamp("event_time"))

windowed_agg = parsed \
    .withWatermark("event_time", "2 minutes") \
    .groupBy(
        F.window("event_time", "5 minutes", "1 minute"),
        "farm_id"
    ) \
    .agg(
        F.avg("soil_moisture").alias("avg_moisture"),
        F.avg("soil_temp").alias("avg_temp"),
        F.avg("soil_ph").alias("avg_ph"),
        F.count("*").alias("sensor_count")
    )

alerts = parsed \
    .filter(F.col("soil_moisture") < 20.0) \
    .select(
        "farm_id", "field_id", "sensor_id",
        "soil_moisture", "event_time",
        F.lit("LOW_MOISTURE_ALERT").alias("alert_type")
    )

print("Stream definitions ready.")

Stream definitions ready.


In [7]:
import threading

def run_producer():
    import json, time
    from kafka import KafkaProducer
    from datetime import datetime

    p = KafkaProducer(
        bootstrap_servers="kafka:9092",
        value_serializer=lambda v: json.dumps(v, default=str).encode("utf-8")
    )

    df_raw = spark.read \
        .option("multiLine", True) \
        .json("file:///home/jovyan/data/agriflow-raw/soil-sensors.json.gz") \
        .select("farm_id", "field_id", "sensor_id",
                "conditions.soil_moisture_pct",
                "conditions.soil_temp_f",
                "nutrients.ph_level") \
        .limit(100) \
        .toPandas()

    time.sleep(15)  # wait for consumer to start first
    print(f"Producer sending {len(df_raw)} events...")

    for _, row in df_raw.iterrows():
        event = {
            "farm_id": str(row["farm_id"]),
            "field_id": str(row["field_id"]),
            "sensor_id": str(row["sensor_id"]),
            "soil_moisture": float(row["soil_moisture_pct"]) if row["soil_moisture_pct"] else 0.0,
            "soil_temp": float(row["soil_temp_f"]) if row["soil_temp_f"] else 0.0,
            "soil_ph": float(row["ph_level"]) if row["ph_level"] else 0.0,
            "event_time": datetime.now().isoformat()
        }
        p.send("sensor-readings", event)
        p.flush()
        time.sleep(0.2)

    print("Producer done.")

# Start producer in background thread
t = threading.Thread(target=run_producer)
t.start()

# Start streams
query_agg = windowed_agg.writeStream \
    .outputMode("update") \
    .format("memory") \
    .queryName("agg_results") \
    .trigger(processingTime="10 seconds") \
    .start()

query_alerts = alerts.writeStream \
    .outputMode("append") \
    .format("memory") \
    .queryName("alert_results") \
    .trigger(processingTime="10 seconds") \
    .start()

print("Waiting 60 seconds for stream to process...")
time.sleep(60)

print("\n=== Windowed Aggregation Results ===")
spark.sql("SELECT * FROM agg_results ORDER BY window, farm_id").show(20, truncate=False)

print("\n=== Low Moisture Alert Results ===")
spark.sql("SELECT * FROM alert_results").show(20, truncate=False)

query_agg.stop()
query_alerts.stop()
t.join()
print("Streaming complete.")

Waiting 60 seconds for stream to process...
Producer sending 100 events...
Producer done.

=== Windowed Aggregation Results ===
+------------------------------------------+-------+------------------+------------------+------------------+------------+
|window                                    |farm_id|avg_moisture      |avg_temp          |avg_ph            |sensor_count|
+------------------------------------------+-------+------------------+------------------+------------------+------------+
|{2026-04-26 16:48:00, 2026-04-26 16:53:00}|FARM-01|24.6              |55.3              |6.17              |1           |
|{2026-04-26 16:48:00, 2026-04-26 16:53:00}|FARM-01|22.0              |60.5              |6.555             |2           |
|{2026-04-26 16:48:00, 2026-04-26 16:53:00}|FARM-02|44.1              |66.8              |6.776666666666666 |3           |
|{2026-04-26 16:48:00, 2026-04-26 16:53:00}|FARM-02|48.4              |62.0              |7.255             |2           |
|{2026-04-2